# Enarau Objective 4 — Recent Vegetation Condition Composite

Builds the recent (non-historical) wet- and dry-season vegetation-condition layer feeding
Objective 4's condition-adjusted Resistance Model C (`wiki/projects/CERK/Enarau/objective-4-connectivity-corridor-assessment.md`,
Sec.3.2 Priority 4 / Sec.5.7 / Sec.6.4). This is deliberately **not** Objective 2's full
historical anomaly/LandTrendr suite — just a single current-period (2022-2025) snapshot,
combining MSAVI2 (productivity), NDMI (moisture), and inverse BSI (bare soil) into one
`condition_score` band per the plan's own formula, for wet season, dry season, and their mean.

Reuses the same `eetools` Sentinel-2 collection builder and `config.PERIODS`/`SEASON_MONTHS`
constants Objective 2 already established — no new index implementation needed (MSAVI2/NDMI/BSI
are already in `config.INDEX_BANDS_COMMON`, computed via eetools' `INDEX_REGISTRY`).

Requires a `.env` file in the repo root with `EE_PROJECT=<your-gee-cloud-project-id>` and Earth
Engine authenticated on this machine (`earthengine authenticate`).

## 1. Setup

In [1]:
import ee
import eetools
import geemap

from eetools.compositing import build_composite
from eetools.constants import S2_SCALE, S2_SR_COLLECTION
from eetools.io import export_image_list_to_drive
from eetools.sensors.sentinel.preprocessing import get_s2_sr_collection
from eetools.vectors import get_sites_geometry, vector_files_to_feature_collection
from eetools.visualization.vis_params import SITES_VIS_PARAMS

In [2]:
try:
    import config
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "config module not found -- run `uv sync` from the repo root to install this "
        "project (including config.py) into the environment."
    ) from exc

In [3]:
eetools.initialize()

In [7]:
# Local aliases for readability -- values are single-sourced in config.py.
CRS = config.PROJECT_CRS
SEASON_MONTHS = config.SEASON_MONTHS
WATER_MASK_HOUSEKEEPING_BANDS = config.WATER_MASK_HOUSEKEEPING_BANDS

CONDITION_EXPORT_FOLDER = config.CONNECTIVITY_CONDITION_EXPORT_FOLDER
CONDITION_PERIOD = config.CONNECTIVITY_CONDITION_PERIOD  # (2022, 2025)
PRODUCTIVITY_INDEX = config.CONNECTIVITY_CONDITION_PRODUCTIVITY_INDEX  # "MSAVI2"
MOISTURE_INDEX = config.CONNECTIVITY_CONDITION_MOISTURE_INDEX  # "NDMI"
BARE_SOIL_INDEX = config.CONNECTIVITY_CONDITION_BARE_SOIL_INDEX  # "BSI"
CONDITION_WEIGHTS = config.CONNECTIVITY_CONDITION_SCORE_WEIGHTS
PCTL_LOW, PCTL_HIGH = config.CONNECTIVITY_CONDITION_PERCENTILE_BOUNDS

CONDITION_INDEX_BANDS = [PRODUCTIVITY_INDEX, MOISTURE_INDEX, BARE_SOIL_INDEX, "NDVI"]

## 2. Study-area boundary & project geometry

In [8]:
site_files = [(site["path"], site["site_id"], site["site_name"]) for site in config.SITES]
sites_fc = vector_files_to_feature_collection(site_files)

# Buffered union of all sites -- one project-wide geometry for the composite/export below,
# not four per-site clips (same convention as Objectives 1-2).
project_geom = get_sites_geometry(sites_fc).buffer(config.STUDY_AREA_BUFFER_M, maxError=10)

## 3. Current-period Sentinel-2 collection

Only `CONDITION_PERIOD` (2022-2025) is needed here -- unlike Objective 2's full-record builder,
this notebook has no historical time series to construct.

In [9]:
requested_indices = CONDITION_INDEX_BANDS + WATER_MASK_HOUSEKEEPING_BANDS
period_start = ee.Date.fromYMD(CONDITION_PERIOD[0], 1, 1)
period_end = ee.Date.fromYMD(CONDITION_PERIOD[1] + 1, 1, 1)

s2_col = get_s2_sr_collection(project_geom, period_start, period_end, indices=requested_indices)

## 4. Wet- and dry-season median composites

Same wet/dry month-window logic as Objective 2's own `get_date_window` (kept notebook-local
rather than shared via eetools, matching that notebook's own convention).

In [10]:
def get_date_window(year_start: int, year_end: int, season: str) -> tuple[ee.Date, ee.Date]:
    """(start, end) ee.Date window for `season` spanning year_start..year_end (inclusive)."""
    start_month, end_month = SEASON_MONTHS[season]
    return ee.Date.fromYMD(year_start, start_month, 1), ee.Date.fromYMD(year_end, end_month + 1, 1)


def build_season_composite(season: str) -> ee.Image:
    """Median composite of CONDITION_INDEX_BANDS over `season` across the full CONDITION_PERIOD
    (a 3-year seasonal median, per the plan's Sec.3.2 Priority 4 recommendation)."""
    start, end = get_date_window(CONDITION_PERIOD[0], CONDITION_PERIOD[1], season)
    subset = s2_col.filterDate(start, end)
    return build_composite(subset, CONDITION_INDEX_BANDS, composite_stat="median").clip(project_geom)


season_composites = {season: build_season_composite(season) for season in ("wet", "dry")}
print(f"Built {len(season_composites)} season composites: {sorted(season_composites)}")

Built 2 season composites: ['dry', 'wet']


## 5. Robust-percentile scaling

Per the plan's Sec.5.6/5.7 guidance ("do not use min-max scaling without inspecting outliers"),
each raw index is clamp-normalized to 0-1 using `p05`/`p95` percentiles computed once over
`project_geom`, mirroring the R-side `slope_scaled` treatment used later in the Objective 4 R
pipeline (`clamp01((x - p05) / (p95 - p05))`).

In [11]:
def percentile_scale(image: ee.Image, band: str) -> ee.Image:
    """Clamp-normalize `band` to 0-1 using PCTL_LOW/PCTL_HIGH percentiles over project_geom."""
    percentiles = image.select(band).reduceRegion(
        reducer=ee.Reducer.percentile([PCTL_LOW, PCTL_HIGH]),
        geometry=project_geom,
        scale=S2_SCALE,
        tileScale=4,
        maxPixels=1e10,
    )
    p_low = ee.Number(percentiles.get(f"{band}_p{PCTL_LOW}"))
    p_high = ee.Number(percentiles.get(f"{band}_p{PCTL_HIGH}"))
    return image.select(band).subtract(p_low).divide(p_high.subtract(p_low)).clamp(0, 1)

## 6. Build `condition_score` per season, plus the wet/dry mean

`condition_score = 0.40*productivity_score + 0.35*moisture_score + 0.25*inverse_bare_soil_score`
(plan Sec.5.7). BSI is inverted after scaling -- high BSI means more bare/degraded ground, so
`inverse_bare_soil_score = 1 - bsi_scaled`.

In [12]:
def build_condition_score(composite: ee.Image) -> ee.Image:
    productivity_score = percentile_scale(composite, PRODUCTIVITY_INDEX).rename("productivity_score")
    moisture_score = percentile_scale(composite, MOISTURE_INDEX).rename("moisture_score")
    bare_soil_scaled = percentile_scale(composite, BARE_SOIL_INDEX)
    inverse_bare_soil_score = ee.Image(1).subtract(bare_soil_scaled).rename("inverse_bare_soil_score")

    condition_score = (
        productivity_score.multiply(CONDITION_WEIGHTS["productivity"])
        .add(moisture_score.multiply(CONDITION_WEIGHTS["moisture"]))
        .add(inverse_bare_soil_score.multiply(CONDITION_WEIGHTS["inverse_bare_soil"]))
        .rename("condition_score")
    )
    return ee.Image.cat(
        [productivity_score, moisture_score, inverse_bare_soil_score, condition_score]
    ).clip(project_geom)


condition_images = {season: build_condition_score(composite) for season, composite in season_composites.items()}

# Combined wet/dry mean -- the doc doesn't mandate one "annual" condition value, but Resistance
# Model C (Sec.7.3) consumes a single condition band; both seasons remain available separately above
# for sensitivity testing (plan Sec.14.1).
condition_images["current"] = (
    condition_images["wet"]
    .add(condition_images["dry"])
    .divide(2)
    .rename(condition_images["wet"].bandNames())
)
print(f"Built condition_score images: {sorted(condition_images)}")

Built condition_score images: ['current', 'dry', 'wet']


## 7. Visual QA

In [ ]:
Map = geemap.Map()
Map.centerObject(project_geom, 12)
Map.addLayer(sites_fc.style(**SITES_VIS_PARAMS), {}, "Study sites")
Map.addLayer(
    condition_images["current"].select("condition_score"),
    {"min": 0, "max": 1, "palette": ["a50026", "fee08b", "1a9850"]},
    "Condition score (current, wet/dry mean)",
)
Map.addLayer(
    condition_images["wet"].select("condition_score"),
    {"min": 0, "max": 1, "palette": ["a50026", "fee08b", "1a9850"]},
    "Condition score (wet)",
    False,
)
Map.addLayer(
    condition_images["dry"].select("condition_score"),
    {"min": 0, "max": 1, "palette": ["a50026", "fee08b", "1a9850"]},
    "Condition score (dry)",
    False,
)
Map

## 8. Export rasters to Google Drive

One multiband GeoTIFF per season (`productivity_score`, `moisture_score`,
`inverse_bare_soil_score`, `condition_score`), plus the wet/dry-mean `current` version.
Same build-list-then-start convention as Objectives 1/2.

In [13]:
EXPORT_TASKS_CONDITION = [
    (
        image,
        f"condition_score_{season}_{CONDITION_PERIOD[0]}_{CONDITION_PERIOD[1]}_project",
        S2_SCALE,
    )
    for season, image in condition_images.items()
]
print(f"{len(EXPORT_TASKS_CONDITION)} condition-composite export tasks staged.")

3 condition-composite export tasks staged.


**Manual step** -- review the count above, then run to start this batch.

In [14]:
started = export_image_list_to_drive(
    EXPORT_TASKS_CONDITION, aoi=project_geom, folder=CONDITION_EXPORT_FOLDER, crs=CRS
)
print(f"Started {len(started)} condition-composite export tasks to Drive folder '{CONDITION_EXPORT_FOLDER}'.")

Started 3 condition-composite export tasks to Drive folder 'CERK_Enarau_Objective4_ConditionComposite'.


## 9. Limitations & next steps

- This is a single current-period (2022-2025) snapshot, not a time series -- it cannot itself
  distinguish a recently-degraded area from one that has always looked this way. Cross-check
  against Objective 2's historical MSAVI2/NDMI/BSI trend maps and LandTrendr events before
  reading a low `condition_score` as recent degradation.
- Percentile bounds (`config.CONNECTIVITY_CONDITION_PERCENTILE_BOUNDS`, currently p05/p95) are a
  starting value per the plan's own suggestion, not yet calibrated -- same caveat as
  `config.DW_HABITAT_THRESHOLDS` before its ground-truth calibration pass.
- MSAVI2 was chosen over NDVI for the productivity component (savanna/sparse-canopy AOI, same
  rationale as Objective 2's wet-season MSAVI2 LandTrendr run) -- revisit if Model C sensitivity
  testing (plan Sec.14.1, 0%/10%/20% condition weight) suggests NDVI behaves materially
  differently here.
- After downloading the exported GeoTIFFs from Drive into `outputs/rasters/connectivity/`, the
  Objective 4 R pipeline resamples/aggregates these 10 m rasters onto the 30 m connectivity
  master grid -- no aggregation happens in this notebook.